# 강의계획서 자동화 (Colab)

강사용 입력 엑셀을 마스터 템플릿 PPTX 초안으로 자동 변환합니다. **여러 파일을 한 번에** 처리할 수 있습니다.
위에서부터 **순서대로 실행**하세요.

**사전 준비 (한 번만)**: `lecture-plan-automation` 폴더(`src`, `config`, `templates` 포함)를 **내 드라이브(MyDrive)** 에 업로드해 두세요.
→ drive.google.com 에서 폴더째로 드래그. 드라이브에 두면 런타임이 끊겨도 사라지지 않습니다.

입력 엑셀은 드라이브에 올릴 필요 없이 **3번 셀에서 그때그때 업로드**(여러 개 동시 가능)합니다.

**강사 사진(선택)**: `teacher_photos/` 폴더에 사진을 넣어두면 우상단 박스에 자동으로 들어갑니다. `강사명.jpg` 또는 샘플학원 배포 파일명(`academy_과목_강사명.png`) 그대로 인식하며, 엑셀의 강사명과 이름이 같으면 매칭됩니다. 없으면 회색 박스로 둡니다.


## 1. 패키지 설치


In [ ]:
!pip -q install pandas openpyxl python-pptx

## 2. 드라이브 연결 + 프로젝트 폴더 찾기
팝업이 뜨면 계정 인증을 진행하세요. `src 존재: True`, `템플릿 존재: True`로 나오면 정상입니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob

PROJECT_DIR = '/content/drive/MyDrive/lecture-plan-automation'  # 드라이브 내 폴더 경로 (다르면 수정)

# 경로가 틀려도 내 드라이브에서 프로젝트를 자동 탐색(1~2단계 깊이)
if not os.path.exists(os.path.join(PROJECT_DIR, 'src', 'pipeline.py')):
    cand = (glob.glob('/content/drive/MyDrive/*/src/pipeline.py')
            + glob.glob('/content/drive/MyDrive/*/*/src/pipeline.py'))
    if cand:
        PROJECT_DIR = os.path.dirname(os.path.dirname(cand[0]))

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)   # 'src'/'config' 패키지 import 가능하게

print('작업 폴더:', os.getcwd())
print('src 존재:', os.path.exists('src/pipeline.py'))
print('템플릿 존재:', os.path.exists('templates/강의계획서_마스터템플릿.pptx'))

## 3. 강사 입력 엑셀 업로드 (여러 개 동시 가능)
파일 선택창에서 **Ctrl(또는 Shift) 클릭으로 여러 개를 한꺼번에** 선택하세요. 수십 개도 됩니다.


In [ ]:
from google.colab import files

uploaded = files.upload()   # 엑셀(.xlsx) 여러 개 선택 가능
INPUT_FILES = sorted(os.path.abspath(f) for f in uploaded if f.lower().endswith('.xlsx'))
print(f'업로드된 엑셀 {len(INPUT_FILES)}개:')
for f in INPUT_FILES:
    print('  -', os.path.basename(f))

## 4. 실행 (파일별로 각각 PPTX 생성)
업로드한 엑셀마다 `output/<파일명>/` 폴더에 PPTX·정규화 데이터·검증 리포트를 만듭니다.
FAQ 제거 · 박스 자동 확장 · 세로 중앙 정렬은 모두 자동 적용됩니다.


In [ ]:
import unicodedata
from src.pipeline import run_pipeline

def nfc(s):
    return unicodedata.normalize('NFC', s)   # 한글 자모분리(NFD) → 정상 결합(NFC)

results = []
for path in INPUT_FILES:
    name = nfc(os.path.splitext(os.path.basename(path))[0])
    try:
        r = run_pipeline(input_path=path, output_dir=os.path.join('output', name),
                         base_year=2026, make_pptx=True)
        results.append((name, r))
        print(f'[O] {name} : 강좌 {r["lecture_count"]}개')
    except Exception as e:
        print(f'[X] {name} : 오류 - {e}')

print(f'
총 {len(results)}/{len(INPUT_FILES)}개 파일 처리 완료')

## 5. (선택) 결과 미리보기
특정 파일 하나만 이미지로 확인합니다. `PREVIEW_INDEX`를 바꾸면 다른 파일을 봅니다(0=첫 번째). 전체는 6번에서 zip으로 받으세요.
처음 1회 LibreOffice 설치에 1~2분 걸립니다.


In [ ]:
PREVIEW_INDEX = 0

!apt-get -qq install -y libreoffice poppler-utils >/dev/null
!pip -q install pdf2image
from pdf2image import convert_from_path

name, r = results[PREVIEW_INDEX]
pptx = r['pptx_path']
print('미리보기:', name)
!libreoffice --headless --convert-to pdf --outdir /tmp "{pptx}" >/dev/null
pdf = sorted(glob.glob('/tmp/*.pdf'), key=os.path.getmtime)[-1]
for img in convert_from_path(pdf, dpi=120):
    display(img)

## 6. 결과 다운로드 (강사별 폴더로 정리된 zip 한 개)
강사(파일)마다 폴더 하나에 **PPTX·정규화데이터·검증리포트를 모아** zip으로 묶어 내려받습니다.
한글 폴더/파일명은 NFC로 정규화해 Windows 압축 풀기에서도 안 깨집니다.


In [ ]:
import shutil, unicodedata
from google.colab import files

def nfc(s):
    return unicodedata.normalize('NFC', s)

BUNDLE = '강의계획서_결과'
if os.path.exists(BUNDLE):
    shutil.rmtree(BUNDLE)

# 강사별 폴더에 결과 파일을 한데 모음(이름은 NFC로 정규화, pptx는 평탄화)
for name, r in results:
    dest = os.path.join(BUNDLE, nfc(name))
    os.makedirs(dest, exist_ok=True)
    for key in ['pptx_path', 'normalized_xlsx', 'normalized_json', 'validation_report']:
        p = r.get(key)
        if p and os.path.exists(p):
            shutil.copy2(p, os.path.join(dest, nfc(os.path.basename(p))))

if os.path.exists(BUNDLE + '.zip'):
    os.remove(BUNDLE + '.zip')
shutil.make_archive(BUNDLE, 'zip', '.', BUNDLE)   # zip 안에 '강의계획서_결과/<강사>/...' 단일 최상위 폴더
print('압축 완료:', BUNDLE + '.zip  (강사 폴더', len(results), '개)')
files.download(BUNDLE + '.zip')